# Diffusion segmenter — samples, uncertainty, and the denoising trajectory

Loads the trained diffusion model (`python train.py --model diffusion` must have been run) and, for a few images, shows:

1. the image, the classical reference label, the **mean** of several generated masks, and their **std** (experimental)
2. the **reverse-diffusion trajectory**: the mask estimate going from pure noise to the final mask
3. how the Dice against the reference changes with the number of averaged samples

Run it from the project root (same folder as `config.py`). The reference label is the classical one, **not ground truth**.

In [1]:
import os, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

import config
from dataset import load_tif_image, preprocess_image
from models.diffusion_seg import build_diffusion

N_SHOW   = 3                                 # how many images to show
STEPS    = config.DIFFUSION_SAMPLING_STEPS   # denoising steps per sample
N_SAMPLE = config.DIFFUSION_NUM_SAMPLES      # masks averaged

ckpt = os.path.join(config.CHECKPOINT_DIR, config.MODEL_REGISTRY["diffusion"]["checkpoint_name"])
if not os.path.exists(ckpt):
    raise FileNotFoundError(f"{ckpt} not found - run:  python train.py --model diffusion")
model = build_diffusion()          # same architecture/size as training, read from config.py
model.load_weights(ckpt)
print(f"loaded {ckpt}  ({model.count_params():,} parameters), image size {config.IMAGE_SIZE}")

paths = sorted(glob.glob(os.path.join(config.RAW_IMAGE_DIR, "*.tif")) + glob.glob(os.path.join(config.RAW_IMAGE_DIR, "*.tiff")))[:N_SHOW]
print(len(paths), "image(s):", [os.path.basename(p) for p in paths])

def dice(p, g):
    p, g = p.astype(bool), g.astype(bool)
    return (2.0 * (p & g).sum() + 1e-6) / (p.sum() + g.sum() + 1e-6)

I0000 00:00:1793126969.576294    3693 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'utils'

## 1 & 2. Mean prediction, spread, and the noise → mask trajectory

In [ ]:
results = {}
for path in paths:
    name = os.path.basename(path)
    img = cv2.resize(load_tif_image(path), config.IMAGE_SIZE, interpolation=cv2.INTER_LINEAR)
    ref = preprocess_image(img)["edge_mask"]
    prob, std, traj = model.sample(img[None, ..., None], steps=STEPS, n_samples=N_SAMPLE,
                                   seed=config.RANDOM_SEED, return_trajectory=True)
    prob, std = prob[0], std[0]
    pred = (prob >= config.SEGMENTATION_PROB_THRESHOLD).astype(np.uint8)
    results[name] = (img, ref)

    fig, ax = plt.subplots(1, 5, figsize=(21, 4.4))
    ax[0].imshow(img, cmap="gray");                  ax[0].set_title(f"{name} original")
    ax[1].imshow(ref, cmap="gray");                  ax[1].set_title("classical reference label")
    ax[2].imshow(prob, cmap="gray", vmin=0, vmax=1); ax[2].set_title(f"mean of {N_SAMPLE} samples")
    ax[3].imshow(pred, cmap="gray");                 ax[3].set_title(f"thresholded  (Dice vs ref {dice(pred, ref):.3f})")
    im = ax[4].imshow(std, cmap="magma");            ax[4].set_title("std across samples (experimental)")
    plt.colorbar(im, ax=ax[4], fraction=0.046)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

    k = min(6, len(traj)); idx = np.linspace(0, len(traj) - 1, k).round().astype(int)
    fig, ax = plt.subplots(1, k, figsize=(3.2 * k, 3.4))
    for a, i in zip(np.atleast_1d(ax), idx):
        a.imshow(traj[i], cmap="gray", vmin=0, vmax=1); a.set_title(f"step {i + 1}/{len(traj)}"); a.axis("off")
    plt.suptitle(f"{name}: mask estimate from noise (left) to final (right), first sample", y=1.02)
    plt.tight_layout(); plt.show()

## 3. Does averaging more samples help?

In [ ]:
import pandas as pd
rows = []
for name, (img, ref) in results.items():
    row = {"image": name}
    for n in sorted({1, 2, N_SAMPLE}):
        p, _ = model.sample(img[None, ..., None], steps=STEPS, n_samples=n, seed=config.RANDOM_SEED)
        row[f"Dice, {n} sample(s)"] = round(float(dice(p[0] >= config.SEGMENTATION_PROB_THRESHOLD, ref)), 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## Reading this
* A diffusion model *generates* masks, so each run differs; averaging several runs stabilises the result (SegDiff merges multiple samples for the same reason).
* **Dice is against the classical label, not ground truth** — it measures agreement with the weak label.
* The std map is experimental: in a synthetic test it was *not* higher on the object than on background. Check whether it means anything on your images before using it.
* Sampling costs `STEPS x N_SAMPLE` network passes per image; lower `config.DIFFUSION_SAMPLING_STEPS` / `DIFFUSION_NUM_SAMPLES` for speed.